In [ ]:
# Cell 1 — Install verification dependencies
%pip -q install "transformers==4.57.1" accelerate safetensors gdown soundfile

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 639.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 27.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [ ]:
# Cell 2 — Download the publicly shared Kabyle-XLS-R export
from pathlib import Path
import gdown

MODEL_DIR = Path("/content/kabyle_xlsr_public_download")

downloaded = gdown.download_folder(
    url="https://drive.google.com/drive/folders/1eV-XZudIVLUkX5BMRV4YXjVOJsk4VwTH",
    output=str(MODEL_DIR),
    quiet=False,
    use_cookies=False,
)

assert downloaded, "The public folder download did not complete."
assert (MODEL_DIR / "model.safetensors").is_file(), "Weights missing."

print("Downloaded files:")
for path in sorted(MODEL_DIR.iterdir()):
    print(path.name, path.stat().st_size, "bytes")

Retrieving folder contents


Processing file 16d5wI80rrxUvfF2W4F4H6PXAH79SmsTV config.json
Processing file 1iU8iTCK_EM2pY8FZt-ho8OfgKjrBLYe_ model.safetensors
Processing file 1I1AtnPGUOoSkmKaDo1vD7iXHWUhsS0uh preprocessor_config.json
Processing file 1mdS0mOXOatY79_5BOV5R8zYz86OGLwKI special_tokens_map.json
Processing file 1Lt3hkLhJ-tSU10GquCLwlE68Db3mBvNi tokenizer_config.json
Processing file 1h7IHCLye0FLxnwu0wCnG1qaUIuBeYh8S training_args.bin
Processing file 1enn3Goi5XNDAlAZ2b69SQAWJlnShzew7 vocab.json


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=16d5wI80rrxUvfF2W4F4H6PXAH79SmsTV
To: /content/kabyle_xlsr_public_download/config.json
100%|██████████| 2.00k/2.00k [00:00<00:00, 5.14MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1iU8iTCK_EM2pY8FZt-ho8OfgKjrBLYe_
From (redirected): https://drive.google.com/uc?id=1iU8iTCK_EM2pY8FZt-ho8OfgKjrBLYe_&confirm=t&uuid=d426f50b-da2d-4c05-bec4-09167076b4eb
To: /content/kabyle_xlsr_public_download/model.safetensors
100%|██████████| 1.26G/1.26G [00:22<00:00, 56.4MB/s]
Downloading...
From: https://drive.google.com/uc?id=1I1AtnPGUOoSkmKaDo1vD7iXHWUhsS0uh
To: /content/kabyle_xlsr_public_download/preprocessor_config.json
100%|██████████| 256/256 [00:00<00:00, 701kB/s]
Downloading...
From: https://drive.google.com/uc?id=1mdS0mOXOatY79_5BOV5R8zYz86OGLwKI
To: /content/kabyle_xlsr_public_download/special_tokens_map.json
100%|██

Downloaded files:
config.json 2003 bytes
model.safetensors 1261942688 bytes
preprocessor_config.json 256 bytes
special_tokens_map.json 51 bytes
tokenizer_config.json 790 bytes
training_args.bin 5969 bytes
vocab.json 384 bytes



Download completed


In [ ]:
# Cell 3 — Verify model loading, tokenizer compatibility and a forward pass
import torch
from transformers import AutoModelForCTC, Wav2Vec2Processor

processor = Wav2Vec2Processor.from_pretrained(
    str(MODEL_DIR), local_files_only=True
)

model, loading_info = AutoModelForCTC.from_pretrained(
    str(MODEL_DIR),
    local_files_only=True,
    use_safetensors=True,
    output_loading_info=True,
)
model.eval()

assert not loading_info.get("missing_keys"), loading_info
assert not loading_info.get("unexpected_keys"), loading_info
assert not loading_info.get("mismatched_keys"), loading_info

tokenizer = processor.tokenizer

assert len(tokenizer) == model.config.vocab_size == 34
assert model.config.pad_token_id == tokenizer.pad_token_id
assert processor.feature_extractor.sampling_rate == 16000

# Synthetic silence checks execution only, not recognition accuracy.
with torch.inference_mode():
    logits = model(input_values=torch.zeros(1, 16000)).logits

assert logits.shape[-1] == 34
assert torch.isfinite(logits).all().item()

print("PASS: downloaded model loads without missing or unexpected weights.")
print("PASS: 34-token vocabulary and CTC blank ID agree.")
print("PASS: forward pass produces finite logits.")
print("Recognition accuracy and equality to the original checkpoint were not tested.")

PASS: downloaded model loads without missing or unexpected weights.
PASS: 34-token vocabulary and CTC blank ID agree.
PASS: forward pass produces finite logits.
Recognition accuracy and equality to the original checkpoint were not tested.


In [ ]:
# Cell 4 — Release the previous model and download OmniASR
import gc
from pathlib import Path
import gdown

if "model" in globals():
    del model
if "processor" in globals():
    del processor
gc.collect()

OMNI_DIR = Path("/content/omniasr_public_download")

downloaded = gdown.download_folder(
    url="https://drive.google.com/drive/folders/1zW8AQzJq4aNbTWnuUjKzy640OpEFnOgr",
    output=str(OMNI_DIR),
    quiet=False,
    use_cookies=False,
)

assert downloaded, "Public download did not complete."
assert (OMNI_DIR / "model.safetensors").is_file(), "Weights missing."

print("OmniASR export downloaded.")

Retrieving folder contents


Processing file 1wDAJPF0eBqQtlOKLroKmbG6SB-KJ2GiF config.json
Processing file 1ye6b9ddxw1Os6BvzWwjT7bBAJq6K11CZ model.safetensors
Processing file 1mTMKcI8oQan0ZKDqQru1EuAiqPopIgJM preprocessor_config.json
Processing file 1v0xMFsKU_Ij-LidotODcR1nCIDqtYEcE special_tokens_map.json
Processing file 1NP5jeUvSKbvWMaXVOsHM1uzFQSF7G_wK tokenizer_config.json
Processing file 1OAnciBLrO8bMkh3UYQn8GsqtRh2afWEi training_args.bin
Processing file 1jg1R293QE1dNg8xEhMK7-Kjy2HtTB1be vocab.json


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1wDAJPF0eBqQtlOKLroKmbG6SB-KJ2GiF
To: /content/omniasr_public_download/config.json
100%|██████████| 2.09k/2.09k [00:00<00:00, 7.75MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1ye6b9ddxw1Os6BvzWwjT7bBAJq6K11CZ
From (redirected): https://drive.google.com/uc?id=1ye6b9ddxw1Os6BvzWwjT7bBAJq6K11CZ&confirm=t&uuid=bdbacb77-98cc-4887-90a6-ecf4b373588e
To: /content/omniasr_public_download/model.safetensors
100%|██████████| 1.26G/1.26G [00:16<00:00, 74.9MB/s]
Downloading...
From: https://drive.google.com/uc?id=1mTMKcI8oQan0ZKDqQru1EuAiqPopIgJM
To: /content/omniasr_public_download/preprocessor_config.json
100%|██████████| 256/256 [00:00<00:00, 911kB/s]
Downloading...
From: https://drive.google.com/uc?id=1v0xMFsKU_Ij-LidotODcR1nCIDqtYEcE
To: /content/omniasr_public_download/special_tokens_map.json
100%|██████████| 51.0/5

OmniASR export downloaded.



Download completed


In [ ]:
# Cell 5 — Verify the downloaded OmniASR export
import torch
from transformers import AutoModelForCTC, Wav2Vec2Processor

processor = Wav2Vec2Processor.from_pretrained(
    str(OMNI_DIR), local_files_only=True
)

model, loading_info = AutoModelForCTC.from_pretrained(
    str(OMNI_DIR),
    local_files_only=True,
    use_safetensors=True,
    output_loading_info=True,
)
model.eval()

assert not loading_info.get("missing_keys"), loading_info
assert not loading_info.get("unexpected_keys"), loading_info
assert not loading_info.get("mismatched_keys"), loading_info

tokenizer = processor.tokenizer
assert len(tokenizer) == model.config.vocab_size == 34
assert model.config.pad_token_id == tokenizer.pad_token_id
assert processor.feature_extractor.sampling_rate == 16000

with torch.inference_mode():
    logits = model(input_values=torch.zeros(1, 16000)).logits

assert logits.shape[-1] == 34
assert torch.isfinite(logits).all().item()

print("PASS: OmniASR public export loads without missing or unexpected weights.")
print("PASS: 34-token vocabulary and CTC blank ID agree.")
print("PASS: forward pass produces finite logits.")
print("Recognition accuracy and equality to checkpoint-660 were not tested.")

PASS: OmniASR public export loads without missing or unexpected weights.
PASS: 34-token vocabulary and CTC blank ID agree.
PASS: forward pass produces finite logits.
Recognition accuracy and equality to checkpoint-660 were not tested.


In [ ]:
# Cell 6 — Release OmniASR and download the Fadhma export
import gc
from pathlib import Path
import gdown

if "model" in globals():
    del model
if "processor" in globals():
    del processor
gc.collect()

FADHMA_DIR = Path("/content/fadhma_public_download")

downloaded = gdown.download_folder(
    url="https://drive.google.com/drive/folders/1-qxxkVcUtmmBr903UJDNdNKbbuKgAHD0",
    output=str(FADHMA_DIR),
    quiet=False,
    use_cookies=False,
)

assert downloaded, "Public download did not complete."
assert (FADHMA_DIR / "model.safetensors").is_file(), "Weights missing."

print("Fadhma export downloaded.")

Retrieving folder contents


Processing file 1hcMz8PizBPLJ-KkVC1J5YWmcnk1juLce config.json
Processing file 1sWcV-kYe3VpK-2yEp8JcvFRPoJF9cXEr model.safetensors
Processing file 1c8sza8Y2npsqEgYAoIOzokqerxxmxbVb preprocessor_config.json
Processing file 1UzoH9l_GUQvWfZoXPufAzGxCvsvSg7Uc special_tokens_map.json
Processing file 1-uCzJY_c_yCOnszJOXXMAiKECE0tB4r9 tokenizer_config.json
Processing file 1zoyriy9gfXGsFn9eQdc3lQc7jfBkgqsw training_args.bin
Processing file 1ByS6YljWyzyzwmezkKf-uZazFD9y8ZKY vocab.json


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1hcMz8PizBPLJ-KkVC1J5YWmcnk1juLce
To: /content/fadhma_public_download/config.json
100%|██████████| 2.08k/2.08k [00:00<00:00, 8.22MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1sWcV-kYe3VpK-2yEp8JcvFRPoJF9cXEr
From (redirected): https://drive.google.com/uc?id=1sWcV-kYe3VpK-2yEp8JcvFRPoJF9cXEr&confirm=t&uuid=fb292e45-b311-47cc-b4bf-8ba5715b850d
To: /content/fadhma_public_download/model.safetensors
100%|██████████| 1.26G/1.26G [00:15<00:00, 83.0MB/s]
Downloading...
From: https://drive.google.com/uc?id=1c8sza8Y2npsqEgYAoIOzokqerxxmxbVb
To: /content/fadhma_public_download/preprocessor_config.json
100%|██████████| 256/256 [00:00<00:00, 752kB/s]
Downloading...
From: https://drive.google.com/uc?id=1UzoH9l_GUQvWfZoXPufAzGxCvsvSg7Uc
To: /content/fadhma_public_download/special_tokens_map.json
100%|██████████| 51.0/51.0 

Fadhma export downloaded.



Download completed


In [ ]:
# Cell 7 — Verify the downloaded Fadhma export
import torch
from transformers import AutoModelForCTC, Wav2Vec2Processor

processor = Wav2Vec2Processor.from_pretrained(
    str(FADHMA_DIR), local_files_only=True
)

model, loading_info = AutoModelForCTC.from_pretrained(
    str(FADHMA_DIR),
    local_files_only=True,
    use_safetensors=True,
    output_loading_info=True,
)
model.eval()

assert not loading_info.get("missing_keys"), loading_info
assert not loading_info.get("unexpected_keys"), loading_info
assert not loading_info.get("mismatched_keys"), loading_info

tokenizer = processor.tokenizer
assert len(tokenizer) == model.config.vocab_size == 34
assert model.config.pad_token_id == tokenizer.pad_token_id
assert processor.feature_extractor.sampling_rate == 16000

with torch.inference_mode():
    logits = model(input_values=torch.zeros(1, 16000)).logits

assert logits.shape[-1] == 34
assert torch.isfinite(logits).all().item()

print("PASS: Fadhma public export loads without missing or unexpected weights.")
print("PASS: 34-token vocabulary and CTC blank ID agree.")
print("PASS: forward pass produces finite logits.")
print("Recognition accuracy and equality to checkpoint-330 were not tested.")

PASS: Fadhma public export loads without missing or unexpected weights.
PASS: 34-token vocabulary and CTC blank ID agree.
PASS: forward pass produces finite logits.
Recognition accuracy and equality to checkpoint-330 were not tested.


In [ ]:
# Cell 8 — Release Fadhma and download the selected MMS checkpoint
import gc
from pathlib import Path
import gdown

if "model" in globals():
    del model
if "processor" in globals():
    del processor
gc.collect()

MMS_DIR = Path("/content/mms_half_spk009_public_download")

downloaded = gdown.download_folder(
    url="https://drive.google.com/drive/folders/1bMm_8hWL0ObC2pqmJ4Yhos4peZhJ8lqt",
    output=str(MMS_DIR),
    quiet=False,
    use_cookies=False,
)

assert downloaded, "Public download did not complete."
assert (MMS_DIR / "model.safetensors").is_file(), "Weights missing."

print("MMS checkpoint downloaded.")

Retrieving folder contents


Processing file 1aMeZnHrOznDmoBpvLuCB_21ojssAIee3 config.json
Processing file 1Uo8YzMX-d0rR-_jOyltP_FNzIOLfmBcb model.safetensors
Processing file 1rgqsOitPrJ7qnAveH2cI5xP7BQ5CnFST optimizer.pt
Processing file 158KUiiWIKfQIiSkFceWSnRxpg2N27DSz preprocessor_config.json
Processing file 1o5bhD51aoA_8_CtGvE7gW_J3ojUDT755 rng_state.pth
Processing file 1SvZuXcu1iMK4U1ohNnk-MdvPhTJezsrS scaler.pt
Processing file 1uBD5zkDq1x40HmvlVVn-CGKjPfOofhpO scheduler.pt
Processing file 13-hqyA-8H-VECOP8peWf5fi52gt24UHN special_tokens_map.json
Processing file 1rm6xu2RomBw0KVCB7a_V-dwdwHN_tS9B tokenizer_config.json
Processing file 1V7a6lxKROHc6WlQERrt1IghejwxVPsaC trainer_state.json
Processing file 1RwZ3G6dK3SfuWKyrxfPOeK30ifcq2pnZ training_args.bin
Processing file 1KCA498ZWC6r5QQ4HnLp8t-zjRKcTM7jV vocab.json


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1aMeZnHrOznDmoBpvLuCB_21ojssAIee3
To: /content/mms_half_spk009_public_download/config.json
100%|██████████| 2.00k/2.00k [00:00<00:00, 7.30MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1Uo8YzMX-d0rR-_jOyltP_FNzIOLfmBcb
From (redirected): https://drive.google.com/uc?id=1Uo8YzMX-d0rR-_jOyltP_FNzIOLfmBcb&confirm=t&uuid=39c2b7af-8468-4fbb-be21-2f78717485e4
To: /content/mms_half_spk009_public_download/model.safetensors
100%|██████████| 3.86G/3.86G [01:03<00:00, 60.9MB/s]
Downloading...
From: https://drive.google.com/uc?id=1rgqsOitPrJ7qnAveH2cI5xP7BQ5CnFST
To: /content/mms_half_spk009_public_download/optimizer.pt
100%|██████████| 17.8M/17.8M [00:00<00:00, 85.4MB/s]
Downloading...
From: https://drive.google.com/uc?id=158KUiiWIKfQIiSkFceWSnRxpg2N27DSz
To: /content/mms_half_spk009_public_download/preprocessor_config.js

MMS checkpoint downloaded.



Download completed


In [ ]:
# Cell 9 — Verify the downloaded MMS checkpoint
import torch
from transformers import AutoModelForCTC, Wav2Vec2Processor

processor = Wav2Vec2Processor.from_pretrained(
    str(MMS_DIR), local_files_only=True
)

model, loading_info = AutoModelForCTC.from_pretrained(
    str(MMS_DIR),
    local_files_only=True,
    use_safetensors=True,
    low_cpu_mem_usage=True,
    output_loading_info=True,
)
model.eval()

assert not loading_info.get("missing_keys"), loading_info
assert not loading_info.get("unexpected_keys"), loading_info
assert not loading_info.get("mismatched_keys"), loading_info

tokenizer = processor.tokenizer
assert len(tokenizer) == model.config.vocab_size == 34
assert model.config.pad_token_id == tokenizer.pad_token_id
assert processor.feature_extractor.sampling_rate == 16000

with torch.inference_mode():
    logits = model(input_values=torch.zeros(1, 16000)).logits

assert logits.shape[-1] == 34
assert torch.isfinite(logits).all().item()

print("PASS: MMS checkpoint-524 loads without missing or unexpected weights.")
print("PASS: 34-token vocabulary and CTC blank ID agree.")
print("PASS: forward pass produces finite logits.")
print("Recognition accuracy and equality to the original file were not tested.")

PASS: MMS checkpoint-524 loads without missing or unexpected weights.
PASS: 34-token vocabulary and CTC blank ID agree.
PASS: forward pass produces finite logits.
Recognition accuracy and equality to the original file were not tested.


In [ ]:
# Cell 10A — Show which adapter files downloaded successfully
from pathlib import Path

folder = Path("/content/mms_full_adapter_public_download")

if folder.exists():
    for path in sorted(folder.rglob("*")):
        if path.is_file():
            print(
                path.relative_to(folder),
                f"— {path.stat().st_size:,} bytes"
            )
else:
    print("No download folder was created.")

config/config.json — 1,999 bytes
processor/preprocessor_config.json — 254 bytes
processor/special_tokens_map.json — 51 bytes


In [ ]:
# Cell 10B — Retry the tokenizer configuration download
from pathlib import Path
import gdown
import json

destination = Path(
    "/content/mms_full_adapter_public_download/"
    "processor/tokenizer_config.json"
)

result = gdown.download(
    id="1QZWDBHTdIhD37znRGGlSUfv-myuOrsv6",
    output=str(destination),
    quiet=False,
    use_cookies=False,
)

assert result, "Download failed."
with destination.open() as f:
    json.load(f)

print("PASS: tokenizer_config.json downloaded and contains valid JSON.")

FileURLRetrievalError: Failed to retrieve file url:

	Cannot retrieve the public link of the file. You may need to change
	the permission to 'Anyone with the link', or have had many accesses.
	Check FAQ in https://github.com/wkentaro/gdown?tab=readme-ov-file#faq.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=1QZWDBHTdIhD37znRGGlSUfv-myuOrsv6

but Gdown can't. Please check connections and permissions.

In [2]:
# Cell 10C — Upload the browser-downloaded adapter files to Colab
from google.colab import files
from pathlib import Path
import json

ADAPTER_DIR = Path("/content/mms_full_adapter_public_download")
uploaded = files.upload()

destinations = {
    "tokenizer_config.json": ADAPTER_DIR / "processor/tokenizer_config.json",
    "vocab.json": ADAPTER_DIR / "processor/vocab.json",
    "best_adapter.safetensors": ADAPTER_DIR / "best_adapter.safetensors",
}

missing = set(destinations) - set(uploaded)
assert not missing, f"Select files with these exact names: {sorted(missing)}"

for name, destination in destinations.items():
    data = uploaded[name]
    assert data, f"Empty file: {name}"
    if name.endswith(".json"):
        json.loads(data.decode("utf-8"))
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(data)

required = [
    "config/config.json",
    "processor/preprocessor_config.json",
    "processor/special_tokens_map.json",
    "processor/tokenizer_config.json",
    "processor/vocab.json",
    "best_adapter.safetensors",
]
for name in required:
    path = ADAPTER_DIR / name
    assert path.is_file() and path.stat().st_size > 0, f"Missing: {name}"

print("Required files are present. Run Cell 11 next.")

Saving best_adapter.safetensors to best_adapter.safetensors
Saving vocab.json to vocab (1).json
Saving tokenizer_config.json to tokenizer_config (1).json


AssertionError: Select files with these exact names: ['tokenizer_config.json', 'vocab.json']

In [3]:
# Cell 10D — Restore the expected filenames from the uploaded files
from pathlib import Path
import re
import json

ADAPTER_DIR = Path("/content/mms_full_adapter_public_download")

destinations = {
    "tokenizer_config.json": ADAPTER_DIR / "processor/tokenizer_config.json",
    "vocab.json": ADAPTER_DIR / "processor/vocab.json",
    "best_adapter.safetensors": ADAPTER_DIR / "best_adapter.safetensors",
}

for expected_name, destination in destinations.items():
    matches = [
        name for name in uploaded
        if re.sub(r" \(\d+\)(?=\.[^.]+$)", "", name) == expected_name
    ]
    assert len(matches) == 1, (
        f"Expected one upload for {expected_name}; found {matches}"
    )

    data = uploaded[matches[0]]
    assert data, f"Empty file: {matches[0]}"
    if expected_name.endswith(".json"):
        json.loads(data.decode("utf-8"))

    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(data)
    print("Ready:", destination)

print("Files restored. Run Cell 11 now.")

Ready: /content/mms_full_adapter_public_download/processor/tokenizer_config.json
Ready: /content/mms_full_adapter_public_download/processor/vocab.json
Ready: /content/mms_full_adapter_public_download/best_adapter.safetensors
Files restored. Run Cell 11 now.


In [5]:
# Cell 10E — Check the configuration files required for loading
from pathlib import Path
import json

ADAPTER_DIR = Path("/content/mms_full_adapter_public_download")

required = [
    "config/config.json",
    "processor/preprocessor_config.json",
    "processor/tokenizer_config.json",
    "processor/special_tokens_map.json",
    "processor/vocab.json",
]

for name in required:
    path = ADAPTER_DIR / name
    if not path.is_file():
        print("MISSING:", name)
        continue

    try:
        data = json.loads(path.read_text())
        print("VALID JSON:", name, f"({path.stat().st_size} bytes)")
        if name.endswith("preprocessor_config.json"):
            print("Contents:", data)
    except (ValueError, UnicodeError) as error:
        print("UNREADABLE:", name, str(error))

print("\nOther copies of preprocessor_config.json:")
for path in ADAPTER_DIR.rglob("preprocessor_config.json"):
    print(path)

MISSING: config/config.json
MISSING: processor/preprocessor_config.json
VALID JSON: processor/tokenizer_config.json (790 bytes)
MISSING: processor/special_tokens_map.json
VALID JSON: processor/vocab.json (384 bytes)

Other copies of preprocessor_config.json:


In [6]:
# Cell 10F — Restore the missing original configuration files
from google.colab import files
from pathlib import Path
import json
import re

ADAPTER_DIR = Path("/content/mms_full_adapter_public_download")
config_uploads = files.upload()

destinations = {
    "config.json": ADAPTER_DIR / "config/config.json",
    "preprocessor_config.json":
        ADAPTER_DIR / "processor/preprocessor_config.json",
    "special_tokens_map.json":
        ADAPTER_DIR / "processor/special_tokens_map.json",
}

for expected, destination in destinations.items():
    matches = [
        name for name in config_uploads
        if re.sub(r" \(\d+\)(?=\.[^.]+$)", "", name) == expected
    ]
    assert len(matches) == 1, (
        f"Select exactly one {expected}; found {matches}"
    )

    data = config_uploads[matches[0]]
    json.loads(data.decode("utf-8"))
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(data)
    print("Restored:", destination)

required = [
    "config/config.json",
    "processor/preprocessor_config.json",
    "processor/special_tokens_map.json",
    "processor/tokenizer_config.json",
    "processor/vocab.json",
    "best_adapter.safetensors",
]

for name in required:
    path = ADAPTER_DIR / name
    assert path.is_file() and path.stat().st_size > 0, f"Missing: {name}"

print("All required files are present. Run Cell 11.")

Saving special_tokens_map.json to special_tokens_map.json
Saving preprocessor_config.json to preprocessor_config.json
Saving config.json to config.json
Restored: /content/mms_full_adapter_public_download/config/config.json
Restored: /content/mms_full_adapter_public_download/processor/preprocessor_config.json
Restored: /content/mms_full_adapter_public_download/processor/special_tokens_map.json
All required files are present. Run Cell 11.


In [7]:
# Cell 10F — Restore the missing original configuration files
from google.colab import files
from pathlib import Path
import json
import re

ADAPTER_DIR = Path("/content/mms_full_adapter_public_download")
config_uploads = files.upload()

destinations = {
    "config.json": ADAPTER_DIR / "config/config.json",
    "preprocessor_config.json":
        ADAPTER_DIR / "processor/preprocessor_config.json",
    "special_tokens_map.json":
        ADAPTER_DIR / "processor/special_tokens_map.json",
}

for expected, destination in destinations.items():
    matches = [
        name for name in config_uploads
        if re.sub(r" \(\d+\)(?=\.[^.]+$)", "", name) == expected
    ]
    assert len(matches) == 1, (
        f"Select exactly one {expected}; found {matches}"
    )

    data = config_uploads[matches[0]]
    json.loads(data.decode("utf-8"))
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(data)
    print("Restored:", destination)

required = [
    "config/config.json",
    "processor/preprocessor_config.json",
    "processor/special_tokens_map.json",
    "processor/tokenizer_config.json",
    "processor/vocab.json",
    "best_adapter.safetensors",
]

for name in required:
    path = ADAPTER_DIR / name
    assert path.is_file() and path.stat().st_size > 0, f"Missing: {name}"

print("All required files are present. Run Cell 11.")

Saving special_tokens_map.json to special_tokens_map (1).json
Saving preprocessor_config.json to preprocessor_config (1).json
Saving config.json to config (1).json
Restored: /content/mms_full_adapter_public_download/config/config.json
Restored: /content/mms_full_adapter_public_download/processor/preprocessor_config.json
Restored: /content/mms_full_adapter_public_download/processor/special_tokens_map.json
All required files are present. Run Cell 11.


In [ ]:
# Cell 10 — Release the previous model and download the full-data MMS adapter
import gc
from pathlib import Path
import gdown

if "model" in globals():
    del model
if "processor" in globals():
    del processor
gc.collect()

ADAPTER_DIR = Path("/content/mms_full_adapter_public_download")

downloaded = gdown.download_folder(
    url="https://drive.google.com/drive/folders/1yewRjLwYJLnOujF2NGDLdAV7gn9Zx_Fk",
    output=str(ADAPTER_DIR),
    quiet=False,
    use_cookies=False,
)

assert downloaded, "Public download did not complete."
assert (ADAPTER_DIR / "best_adapter.safetensors").is_file()
assert (ADAPTER_DIR / "config/config.json").is_file()
assert (ADAPTER_DIR / "processor/vocab.json").is_file()

print("Full-data MMS adapter and supporting files downloaded.")

Retrieving folder contents


Retrieving folder 1bgrGSCjI8V645HwZ6xYr7nwbk1gz5BZd config
Processing file 1-druJcFjanPhy0enZAc9zrbilDfZwu9k config.json
Retrieving folder 1n2h3dc3AllmwqvD_PXIgiQ-FLA3shYD1 processor
Processing file 1ELSTNsZbRPzZ45tNhe-3FRrKKLC0rxqM preprocessor_config.json
Processing file 1KT6J-ZM42lZ8fOY24wwQtpUBkUBau4iP special_tokens_map.json
Processing file 1QZWDBHTdIhD37znRGGlSUfv-myuOrsv6 tokenizer_config.json
Processing file 17s0iVXN2yeTH5km_CiRcO3OXrpshibhj vocab.json
Retrieving folder 1E18JXH64WTjem_ZWN0wELBdKLuDyFlK6 trainer_state
Retrieving folder 15geXo0VjoZRLpXUSSLhtYLhGpyYOx-nO checkpoint-440
Processing file 1aRAyp7e6V9bNhBXzFeR1TSA57N9BMnIz config.json
Processing file 1t35PSPSAv5eCyYrXOjC-IaamCDb_-7BV model.safetensors
Processing file 1qSPFGPKt6KYLhHzg_13JW8CuswflwrdB optimizer.pt
Processing file 1iyJplJcbrQXNWfE7dZaBhn4n7v5nQ4Dq preprocessor_config.json
Processing file 1pFdpGi_F1EaIclz1y3NW_7pgWHpNNJpA rng_state.pth
Processing file 1P1wGxZWxikCMuc3PWHj-w3JGOEk_piHt scaler.pt
Processing

Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1-druJcFjanPhy0enZAc9zrbilDfZwu9k
To: /content/mms_full_adapter_public_download/config/config.json
100%|██████████| 2.00k/2.00k [00:00<00:00, 5.34MB/s]
Downloading...
From: https://drive.google.com/uc?id=1ELSTNsZbRPzZ45tNhe-3FRrKKLC0rxqM
To: /content/mms_full_adapter_public_download/processor/preprocessor_config.json
100%|██████████| 254/254 [00:00<00:00, 878kB/s]
Downloading...
From: https://drive.google.com/uc?id=1KT6J-ZM42lZ8fOY24wwQtpUBkUBau4iP
To: /content/mms_full_adapter_public_download/processor/special_tokens_map.json
100%|██████████| 51.0/51.0 [00:00<00:00, 201kB/s]


FileURLRetrievalError: Failed to retrieve file url:

	Cannot retrieve the public link of the file. You may need to change
	the permission to 'Anyone with the link', or have had many accesses.
	Check FAQ in https://github.com/wkentaro/gdown?tab=readme-ov-file#faq.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=1QZWDBHTdIhD37znRGGlSUfv-myuOrsv6

but Gdown can't. Please check connections and permissions.

In [8]:
# Cell 11 — Restore and verify the full-data MMS adapter
import torch
from safetensors.torch import load_file
from transformers import (
    Wav2Vec2Config,
    Wav2Vec2ForCTC,
    Wav2Vec2Processor,
)

processor = Wav2Vec2Processor.from_pretrained(
    str(ADAPTER_DIR / "processor"),
    local_files_only=True,
)
config = Wav2Vec2Config.from_pretrained(
    str(ADAPTER_DIR / "config"),
    local_files_only=True,
)

model, base_loading = Wav2Vec2ForCTC.from_pretrained(
    "facebook/mms-1b-all",
    config=config,
    ignore_mismatched_sizes=True,
    low_cpu_mem_usage=True,
    output_loading_info=True,
)

adapter_state = load_file(
    str(ADAPTER_DIR / "best_adapter.safetensors")
)

# Require every adapter parameter and both trained CTC-head tensors.
expected_adapter_keys = set(model._get_adapters())
assert expected_adapter_keys, "No adapter parameters identified."
assert expected_adapter_keys.issubset(adapter_state), (
    "Missing adapter tensors:",
    sorted(expected_adapter_keys - set(adapter_state)),
)
assert {"lm_head.weight", "lm_head.bias"}.issubset(adapter_state)

# Any parameters not loaded from the base must be supplied by the adapter.
unresolved = set(base_loading.get("missing_keys", []))
for entry in base_loading.get("mismatched_keys", []):
    unresolved.add(entry if isinstance(entry, str) else entry[0])
assert not (unresolved - set(adapter_state)), (
    "Unrestored parameters:", sorted(unresolved - set(adapter_state))
)
assert not base_loading.get("error_msgs"), base_loading

result = model.load_state_dict(adapter_state, strict=False)
assert not result.unexpected_keys, result.unexpected_keys

# Backbone keys are expected to be absent from an adapter-only file.
restored = model.state_dict()
assert all(
    torch.equal(restored[name].cpu(), value)
    for name, value in adapter_state.items()
), "An adapter tensor was not restored exactly."

tokenizer = processor.tokenizer
assert len(tokenizer) == model.config.vocab_size == 34
assert model.config.pad_token_id == tokenizer.pad_token_id
assert processor.feature_extractor.sampling_rate == 16000

model.eval()
with torch.inference_mode():
    logits = model(input_values=torch.zeros(1, 16000)).logits

assert logits.shape[-1] == 34
assert torch.isfinite(logits).all().item()

print("PASS: public adapter download restores onto the MMS backbone.")
print("PASS: adapter parameters and trained CTC head are present and restored.")
print("PASS: 34-token vocabulary and CTC blank ID agree.")
print("PASS: forward pass produces finite logits.")
print("Recognition accuracy and historical backbone revision were not verified.")

config.json:   0%|          | 0.00/2.04k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.86GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1096 [00:00<?, ?it/s]

[transformers] Wav2Vec2ForCTC LOAD REPORT from: facebook/mms-1b-all
Key            | Status   |                                                                                            
---------------+----------+--------------------------------------------------------------------------------------------
lm_head.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([154, 1280]) vs model:torch.Size([34, 1280])
lm_head.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([154]) vs model:torch.Size([34])            

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


PASS: public adapter download restores onto the MMS backbone.
PASS: adapter parameters and trained CTC head are present and restored.
PASS: 34-token vocabulary and CTC blank ID agree.
PASS: forward pass produces finite logits.
Recognition accuracy and historical backbone revision were not verified.
